# Spatial Auto-Correlation Analysis

In [2]:
from smode_import import *
from parula import parula

# drifters data
SMODE_DATA_folder = '/Users/elise/data/SMODE-data/'
dr_files = sorted(glob(SMODE_DATA_folder+'drifters/2023/'+'SMODE_IOP2_surface_drifter_0*.nc'))
if not dr_files: print('No files found')

# underway ship data
uw = xr.open_dataset(SMODE_DATA_folder+'TSG/'+'IOP2_underway.nc')
uw

# TSG
TSG = xr.open_dataset('/Users/elise/data/SMODE-data/S_MODE_IOP2_SRIDE23_tsg_met_bio.nc')
TSG = TSG.drop_duplicates(dim='time_bio')
TSG = TSG.drop_duplicates(dim='time')
flowrate_time_bio = TSG['flow_rate_lab_tsg'].interp(time=TSG['time_bio'], method='linear')
TSG['flowrate_time_bio'] = (TSG.chlorophyll_ACS.dims, flowrate_time_bio.values)

# function
def interp_ship_on_drifter_ACS(TSG, drifter, resample, dist=1):
        
    # chlorohypll flowrate control
    flowrate_okay = np.where(TSG.flowrate_time_bio>1.)

    # Resample data
    TSG_longitude   = TSG.longitude_bio.resample(time_bio=resample).mean()
    TSG_latitude    = TSG.latitude_bio.resample(time_bio=resample).mean()
    TSG_chlorophyll = TSG.chlorophyll_ACS[flowrate_okay].resample(time_bio=resample).mean()
    
    # Interpolate ship data onto drifter time
    interp_lon = TSG_longitude.interp(time_bio=drifter.time, method='linear')
    interp_lat = TSG_latitude.interp(time_bio=drifter.time, method='linear')
    interp_chl = TSG_chlorophyll.interp(time_bio=drifter.time, method='linear')
    
    # Save chlorophyll values when dist < 1 km
    distances = haversine(drifter.latitude, drifter.longitude, interp_lat, interp_lon) # pairwise distances
    idx = (distances <= dist)
    
    # Logarithmic chlorophyll rate of change
    # dP/dt = P2 * log(P2/P1) / dt
    dChldt_ship = np.empty(len(drifter.time[idx]))*np.nan
    P1,P2 = interp_chl[idx][:-1].values, interp_chl[idx][1:].values
    dChl = np.log(P2/P1)
    dt   = (drifter.time[idx].diff(dim='time')*1e-9).astype(float) # seconds
    #print(dChldt_ship.shape,dChl.shape,dt.shape)
    dt   = dt/86400. # convert to day
    dChldt_ship[1::] = dChl/dt
    
    dataset = xr.Dataset(
    {
        'lon': ('time', drifter.longitude[idx].values),
        'lat': ('time', drifter.latitude[idx].values),
        'chl': ('time', interp_chl[idx].values),
        'dChldt': ('time', dChldt_ship),
        'dist_from_ship': ('time', distances[idx].values),
    },
    coords={'time': drifter.time[idx].values},
    attrs={'name': drifter.title[-7::]},
    )
    
    return dataset
    

In [29]:
distances = np.arange(0.1, 4.1, 0.1)
IDs = ['4696942', '4697436', '4697439', '4697491', '4697532', '4697629', '4697637', '4704406']

results = []

for dist in distances:
    for ID in IDs:
        drifter = xr.open_dataset(f"{SMODE_DATA_folder}/drifters/2023/SMODE_IOP2_surface_drifter_0-{ID}.nc")
        drifter = drifter.isel(time=drifter.position_QCflag==1)
        
        dr = interp_ship_on_drifter_ACS(TSG, drifter, resample='4h', dist=dist)
        if len(dr.time) > 0:
            dr = dr.resample(time='1h').mean()
            chl = dr['chl'].dropna('time')

            # Remove the mean (optional)
            chl_anom = chl - chl.mean()

            # Compute autocorrelation at lag 1 hour
            if len(chl_anom.time) > 1:
                ac1 = (chl_anom * chl_anom.shift(time=1)).mean() / (chl_anom ** 2).mean()
                results.append({'ID': ID, 'distance': dist, 'autocorr': ac1.item()})

df = pd.DataFrame(results)



In [30]:
plt.figure(figsize=(8,5))
sns.lineplot(data=df, x='distance', y='autocorr', ci='sd', marker='o')
plt.axvline(1, color='r', linestyle='--', label='1 km')
plt.xlabel('Interpolation distance (km)')
plt.ylabel('Lag-1 Autocorrelation of CHLOROPHYLL')
plt.title('Autocorrelation vs Interpolation Distance')
plt.legend()
plt.tight_layout()


ModuleNotFoundError: No module named 'seaborn'